# Search engine with embeddings

https://www.nlplanet.org/course-practical-nlp/01-intro-to-nlp/12-search-engine-embeddings

In [1]:
from huggingface_hub import hf_hub_download

import pandas as pd
import numpy as np

from sentence_transformers import SentenceTransformer, util
import torch

Download Dataset

In [2]:
# download dataset of Medium articles from 
# https://huggingface.co/datasets/fabiochiu/medium-articles
df_articles = pd.read_csv(
  hf_hub_download("fabiochiu/medium-articles", repo_type="dataset", filename="medium_articles.csv")
)

# There are 192,368 articles in total, but let's keep only the first 1,000 to
# make computations faster
df_articles = df_articles[:1000].reset_index(drop=True)

df_articles.head()

,title,text,url,authors,timestamp,tags
0,Mental Note Vol. 24,Photo by Josh Riemer on Unsplash\n\nMerry Chri...,https://medium.com/invisible-illness/mental-no...,['Ryan Fan'],2020-12-26 03:38:10.479000+00:00,"['Mental Health', 'Health', 'Psychology', 'Sci..."
1,Your Brain On Coronavirus,Your Brain On Coronavirus\n\nA guide to the cu...,https://medium.com/age-of-awareness/how-the-pa...,['Simon Spichak'],2020-09-23 22:10:17.126000+00:00,"['Mental Health', 'Coronavirus', 'Science', 'P..."
2,Mind Your Nose,Mind Your Nose\n\nHow smell training can chang...,https://medium.com/neodotlife/mind-your-nose-f...,[],2020-10-10 20:17:37.132000+00:00,"['Biotechnology', 'Neuroscience', 'Brain', 'We..."
3,The 4 Purposes of Dreams,Passionate about the synergy between science a...,https://medium.com/science-for-real/the-4-purp...,['Eshan Samaranayake'],2020-12-21 16:05:19.524000+00:00,"['Health', 'Neuroscience', 'Mental Health', 'P..."
4,Surviving a Rod Through the Head,"You’ve heard of him, haven’t you? Phineas Gage...",https://medium.com/live-your-life-on-purpose/s...,['Rishav Sinha'],2020-02-26 00:01:01.576000+00:00,"['Brain', 'Health', 'Development', 'Psychology..."


Download the Model 

In [3]:
# download the sentence embeddings model
embedder = SentenceTransformer('all-MiniLM-L6-v2')

c:\Users\TristramArmour\anaconda3\envs\learning\Lib\site-packages\huggingface_hub\file_download.py:1142: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


Embed Documents and Queries, and Compute Cosine Similarity

In [4]:
# Embed article texts.
# It's slow, but it must be done only once
corpus = df_articles["text"].values
corpus_embeddings = embedder.encode(corpus, convert_to_tensor=True)
print(corpus_embeddings.shape)

c:\Users\TristramArmour\anaconda3\envs\learning\Lib\site-packages\transformers\models\bert\modeling_bert.py:435: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at C:\cb\pytorch_1000000000000\work\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:263.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(


torch.Size([1000, 384])


In [6]:
# embed the query
query = "data science nlp"
query_embedding = embedder.encode(query, convert_to_tensor=True)

# find the article with highest cosine-similarity with the query
def show_results(query_embedding, corpus_embeddings, df_articles, top_k=10):
  cos_scores = util.cos_sim(query_embedding, corpus_embeddings)[0]
  top_results = torch.topk(cos_scores, k=top_k)
  position = 1
  for score, idx in zip(top_results[0], top_results[1]):
      row = df_articles.iloc[idx.item()]
      title = row["title"]
      print(f"{position} [score = {score}]: {title}")
      position += 1

show_results(query_embedding, corpus_embeddings, df_articles)

1 [score = 0.4066562354564667]: Think Ahead: Boost Your BI with AI
2 [score = 0.3981834053993225]: AI for Software Engineering — Industry Landscape (03/Oct/2017)
3 [score = 0.3850368559360504]: This is sloppy and hides intent if you ever reference it more
4 [score = 0.36971038579940796]: What is the data science community’s favourite media source?
5 [score = 0.3696000576019287]: Build a Natural Language Classifier With Bert and Tensorflow
6 [score = 0.35367733240127563]: The 5 Tips to Tell Better Stories with Data
7 [score = 0.339456170797348]: Looking back at the eXplainable Artificial Intelligence
8 [score = 0.33313995599746704]: Platform for success: The Telegraph’s big data transformation
9 [score = 0.3317122757434845]: How I Use Data To Build Better Products
10 [score = 0.3311069905757904]: WIDeText: A Multimodal Deep Learning Framework


In [7]:
# embed the query
query = "how to learn data science"
query_embedding = embedder.encode(query, convert_to_tensor=True)
show_results(query_embedding, corpus_embeddings, df_articles)

1 [score = 0.5054750442504883]: This is sloppy and hides intent if you ever reference it more
2 [score = 0.4575732946395874]: How I Use Data To Build Better Products
3 [score = 0.43672943115234375]: The 5 Tips to Tell Better Stories with Data
4 [score = 0.4199758768081665]: Data Science, Alexander of the Times Ahead
5 [score = 0.4143122136592865]: Best Machine Learning Books — Free and Paid — Editorial Recommendations
6 [score = 0.4062786102294922]: Platform for success: The Telegraph’s big data transformation
7 [score = 0.3996283710002899]: What is the data science community’s favourite media source?
8 [score = 0.3980596661567688]: 5 minutes to understand data storytelling!
9 [score = 0.3648417592048645]: What Are The Benefits Of Cloud Data Warehousing?
10 [score = 0.33949705958366394]: Creating Data Lake & Discovering Your Data Using AWS Lake Formation Capabilities


I still find it strange that the answer to a question should have high cosine similarity. I have a test

In [15]:

from sentence_transformers import SentenceTransformer, util

# Load a pre-trained Sentence-BERT model
model = SentenceTransformer('all-mpnet-base-v2')  # You can choose other models too

# Encode the sentences
query_embedding = model.encode('Where does the lion live?')
answer_embedding = model.encode('zoo')

# Calculate cosine similarity
similarity = util.cos_sim(query_embedding, answer_embedding)

print(f"Cosine Similarity: {similarity.item():.4f}")

Cosine Similarity: 0.4686
